# Where does an encoding help — seen clients or unseen ones?

*Setup cells below are carried from the shared analysis so this notebook runs on its own.*


In [1]:
import os

import polars as pl
from sklearn.metrics import average_precision_score, roc_auc_score

from fraud_detection.core.config import resolve_repo_path
from fraud_detection.core.feature_contract import FeatureContract
from fraud_detection.core.feature_contract.admission import load_admission_rules
from fraud_detection.core.schema import (
    CLIENT_ENTITY_ANCHOR,
    CLIENT_ENTITY_COMPONENTS,
    MODEL_INPUT_TABLE,
    SPLIT_TABLE,
)
from fraud_detection.evaluation.entity_purity import Anchor, EntityKey, seen_entity_flag
from fraud_detection.feature_engineering.derivations import apply_derivations, load_frequency_maps
from fraud_detection.training.data import load_raw_split, prepare_features
from fraud_detection.training.model import train_lightgbm

# Same modules the pipeline runs. That is the point of the layering rule in
# docs/code-structure.md: this notebook cannot measure a different implementation from the
# one that gets promoted, because there is only one.
PROJECT = os.environ["GCP_PROJECT_ID"]
SEEDS = [42, 7, 1337, 2024, 91]

In [2]:
rules = load_admission_rules()
declared = pl.DataFrame(
    [{"name": d.name, "tool": d.tool, "input": d.inputs[0]} for d in rules.derivations]
)
print(declared.group_by("tool").len().sort("len", descending=True))

maps = load_frequency_maps()
summary = pl.DataFrame(
    [
        {"column": c, "levels_in_map": len(t), "one_hot_columns_this_would_need": len(t)}
        for c, t in maps.items()
    ]
).sort("levels_in_map", descending=True)
summary

shape: (3, 2)
┌─────────────────────────┬─────┐
│ tool                    ┆ len │
│ ---                     ┆ --- │
│ str                     ┆ u32 │
╞═════════════════════════╪═════╡
│ one_hot                 ┆ 18  │
│ days_since_to_start_day ┆ 7   │
│ frequency_encode        ┆ 5   │
└─────────────────────────┴─────┘


column,levels_in_map,one_hot_columns_this_would_need
str,i64,i64
"""DeviceInfo""",1176,1176
"""addr1""",202,202
"""id_31""",99,99
"""R_emaildomain""",60,60
"""P_emaildomain""",59,59


In [3]:
from google.cloud import bigquery

client = bigquery.Client(project=PROJECT)
contract = FeatureContract.from_json(resolve_repo_path("references/feature-contract.json").read_text())
tables = {"model_input_table": MODEL_INPUT_TABLE, "split_table": SPLIT_TABLE}
raw = {s: load_raw_split(client, PROJECT, s, **tables) for s in ("train", "val", "test")}

key = EntityKey(columns=CLIENT_ENTITY_COMPONENTS, anchors=(Anchor(CLIENT_ENTITY_ANCHOR),))
seen = {s: seen_entity_flag(raw["train"], f, key).fill_null(False).cast(pl.Boolean)
        for s, f in raw.items()}

derived = {s: apply_derivations(f, rules.derivations) for s, f in raw.items()}
print({s: (raw[s].width, derived[s].width) for s in raw})

{'train': (476, 506), 'val': (476, 506), 'test': (476, 506)}


In [4]:
ONE_HOT = [d.name for d in rules.derivations if d.tool == "one_hot"]
FREQ = [d.name for d in rules.derivations if d.tool == "frequency_encode"]
admitted = [c for c in contract.training_features() if c in derived["train"].columns]

VARIANTS = {
    "baseline": [c for c in admitted if c not in ONE_HOT and c not in FREQ],
    "+ one_hot": [c for c in admitted if c not in FREQ],
    "+ frequency": [c for c in admitted if c not in ONE_HOT],
    "+ both": admitted,
}
{name: len(cols) for name, cols in VARIANTS.items()}

{'baseline': 183, '+ one_hot': 201, '+ frequency': 187, '+ both': 205}

In [5]:
WINNING = {"num_leaves": [96], "learning_rate": [0.05], "feature_fraction": [0.6],
           "bagging_fraction": [0.7], "min_child_samples": [80]}


def split_on(columns, name):
    from fraud_detection.training.data import SplitFrame

    frame = derived[name]
    features = prepare_features(frame).select(columns)
    return SplitFrame(
        features=features,
        labels=frame.get_column("isFraud").cast(pl.Int8),
        amounts=frame.get_column("TransactionAmt").cast(pl.Float64),
        seen_in_train=seen[name],
    )


rows = []
for variant, columns in VARIANTS.items():
    splits = {s: split_on(columns, s) for s in ("train", "val", "test")}
    y = splits["test"].labels.to_numpy()
    for seed in SEEDS:
        m = train_lightgbm(splits["train"], splits["val"], splits["test"],
                           search_space=WINNING, n_iter=1, seed=seed)
        rows.append({
            "variant": variant, "seed": seed, "features": len(columns),
            # Raw scores. `test_roc_auc` in metrics.json is computed on the calibrated
            # probabilities, and the submission carries the raw ones.
            "roc_auc": roc_auc_score(y, m.test_scores),
            "pr_auc": average_precision_score(y, m.test_scores),
            "best_iteration": m.booster.best_iteration,
        })
        print(rows[-1], flush=True)

results = pl.DataFrame(rows)
results.write_parquet("encodings_results.parquet")

{'variant': 'baseline', 'seed': 42, 'features': 183, 'roc_auc': 0.8953245269605286, 'pr_auc': 0.5082398986379394, 'best_iteration': 540}
{'variant': 'baseline', 'seed': 7, 'features': 183, 'roc_auc': 0.8948048238793411, 'pr_auc': 0.5165497234414116, 'best_iteration': 1072}
{'variant': 'baseline', 'seed': 1337, 'features': 183, 'roc_auc': 0.8960790197989519, 'pr_auc': 0.5207995278532331, 'best_iteration': 1059}
{'variant': 'baseline', 'seed': 2024, 'features': 183, 'roc_auc': 0.8967299742816417, 'pr_auc': 0.5243158595785282, 'best_iteration': 1407}
{'variant': 'baseline', 'seed': 91, 'features': 183, 'roc_auc': 0.9011997960109956, 'pr_auc': 0.5180950076785836, 'best_iteration': 760}


KeyboardInterrupt: 

In [ ]:
NOISE_SD = {"roc_auc": 0.0029, "pr_auc": 0.0065}  # measured 2026-08-16, five seeds

agg = results.group_by("variant").agg(
    pl.col("features").first(),
    pl.col("roc_auc").mean().alias("roc_mean"),
    pl.col("roc_auc").std().alias("roc_sd"),
    pl.col("pr_auc").mean().alias("pr_mean"),
    pl.col("pr_auc").std().alias("pr_sd"),
)
base = agg.filter(pl.col("variant") == "baseline").row(0, named=True)

# The mean of n seeds has sd/sqrt(n), so a difference of two such means carries
# sd * sqrt(2/n). That is the bar a delta has to clear to be called anything at all.
bar = {m: NOISE_SD[m] * (2 / len(SEEDS)) ** 0.5 for m in NOISE_SD}
print(f"resolution of this design: ROC-AUC ±{bar['roc_auc']:.4f}, PR-AUC ±{bar['pr_auc']:.4f}")

agg.with_columns(
    (pl.col("roc_mean") - base["roc_mean"]).alias("roc_delta"),
    (pl.col("pr_mean") - base["pr_mean"]).alias("pr_delta"),
).with_columns(
    (pl.col("roc_delta").abs() > bar["roc_auc"]).alias("roc_resolvable"),
    (pl.col("pr_delta").abs() > bar["pr_auc"]).alias("pr_resolvable"),
).sort("roc_mean", descending=True)

In [ ]:
encoded = pl.DataFrame([
    {"name": c.name, "admitted": c.admitted, "rejected_by": c.rejected_by or "—",
     "value": c.rejected_value,
     "family": "one_hot" if c.name in ONE_HOT else "frequency"}
    for c in contract.columns if c.name in set(ONE_HOT) | set(FREQ)
])
print(encoded.group_by(["family", "admitted"]).len().sort(["family", "admitted"]))
encoded.filter(~pl.col("admitted")).sort(["family", "name"])

Two rejections worth expecting, and worth checking against what actually happened:

- `card6_is_debit_or_credit` and `card6_is_charge_card` are 1 on 30 and 15 rows out of
  590,540. They were declared deliberately so the redundancy and drift audits could reject
  them on evidence instead of being filtered out by hand beforehand.
- `card1_freq` counts a value that accumulates over the dataset's lifetime, so it is the
  same shape as the velocity counters PSI already rejects — see the uid-aggregate entry in
  `MEASUREMENTS.md`. If PSI rejects it, that is the check being consistent, not a surprise.

### Where a difference, if any, lands

The overall mean hides the segment split, and the segment split is where the promotion gate
looks. Frequency encoding of a client-ish key should help rows whose value was seen in
training and do nothing for the 65.4% whose was not.

In [ ]:
test_seen = seen["test"].to_numpy()
segment_rows = []
for variant, columns in VARIANTS.items():
    splits = {s: split_on(columns, s) for s in ("train", "val", "test")}
    y = splits["test"].labels.to_numpy()
    for seed in SEEDS[:3]:  # three is enough for a segment contrast; five for the headline
        m = train_lightgbm(splits["train"], splits["val"], splits["test"],
                           search_space=WINNING, n_iter=1, seed=seed)
        for label, mask in (("seen", test_seen), ("unseen", ~test_seen)):
            segment_rows.append({
                "variant": variant, "seed": seed, "segment": label, "rows": int(mask.sum()),
                "pr_auc": average_precision_score(y[mask], m.test_scores[mask]),
            })

pl.DataFrame(segment_rows).group_by(["variant", "segment"]).agg(
    pl.col("rows").first(),
    pl.col("pr_auc").mean().alias("pr_mean"),
    pl.col("pr_auc").std().alias("pr_sd"),
).sort(["segment", "pr_mean"], descending=[False, True])

### What to write down

Fill this in from the cells above, and put the same table in `MEASUREMENTS.md`. The rules,
stated before the numbers exist so they cannot be adjusted to fit them:

| Outcome | What goes in MEASUREMENTS.md |
| --- | --- |
| Delta clears ±0.0018 ROC-AUC | A measured effect, with the seed spread beside it |
| Delta inside the band | **"Unmeasured at five seeds"** — never "no effect" |
| One-hot inside the band | The comment in `training/data.py` stands, and now on evidence rather than on assertion |
| Frequency clears the band | Extend it: `card1_addr1` combinations, `id_33`, and the same discipline |
| Frequency inside the band | The encoding is not the gap. The gap to ~0.96 is transductive uid statistics, which this project refuses on purpose — `docs/adversarial-drift.md` |

The most likely outcome is that everything here lands inside the band. Both encodings are
worth ~0.003 at best on a model this far from the leaderboard, and the noise floor is
0.0029. Recording that plainly is more useful than a fourth retracted entry.